In [ ]:
# # install octave
# !sudo apt-get -qq update
# !sudo apt-get -qq install octave octave-signal liboctave-dev

# # install oct2py that compatible with colab
# import os

# from pkg_resources import get_distribution

# os.system(
#     f"pip install -qq"
#     f" ipykernel=={get_distribution('ipykernel').version}"
#     f" ipython=={get_distribution('ipython').version}"
#     f" tornado=={get_distribution('tornado').version}"
#     f" oct2py"
# )

# # install packages
# !pip install -qq matpower matpowercaseframesa

In [ ]:
import oct2py

import matpower

print(f"oct2py version: {oct2py.__version__}")
print(f"matpower version: {matpower.__version__}")

oct2py version: 5.8.0
matpower version: 8.1.0.2.2.4


In [ ]:
import os

import numpy as np
import pandas as pd
from matpowercaseframes import CaseFrames

from matpower import path_matpower, start_instance

In [ ]:
path = os.path.join(path_matpower, "data/case9.m")

In [ ]:
m = start_instance()

## Original Case

In [ ]:
cf = CaseFrames(path, load_case_engine=m)

mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()

display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,1,4,0.0000,0.0576,0.000,250,250,250,0,0,1,-360,360,71.641021,27.045924,-71.641021,-23.923127
2,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.703670,1.030006,-30.537263,-16.543365
3,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.462737,-13.456635,60.816586,-18.074836
4,3,6,0.0000,0.0586,0.000,300,300,300,0,0,1,-360,360,85.000000,-10.859709,-85.000000,14.955327
5,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.183414,3.119508,-24.095417,-24.295823
6,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.904583,-10.704177,76.379866,-0.797331
7,8,2,0.0000,0.0625,0.000,250,250,250,0,0,1,-360,360,-163.000000,9.178149,163.000000,6.653660
8,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.620134,-8.380817,-84.320163,-11.312751
9,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.679837,-38.687249,40.937352,22.893121


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,1,3,0,0,0,0,1,1.040000,0.000000,345,1,1.1,0.9
2,2,2,0,0,0,0,1,1.025000,9.280005,345,1,1.1,0.9
3,3,2,0,0,0,0,1,1.025000,4.664751,345,1,1.1,0.9
4,4,1,0,0,0,0,1,1.025788,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.012654,-3.687396,345,1,1.1,0.9
6,6,1,0,0,0,0,1,1.032353,1.966716,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.015883,0.727536,345,1,1.1,0.9
8,8,1,0,0,0,0,1,1.025769,3.719701,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995631,-3.988805,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,1,71.641021,27.045924,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,2,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,3,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


## Reduce System (Remove Transformers)

In [ ]:
# --- 1. Split ---
is_tx = cf.branch["BR_R"] == 0
tx = cf.branch[is_tx][["F_BUS", "T_BUS"]].astype(int)
cf.branch = cf.branch[~is_tx].copy()

# --- 2. Build remap ---
used = set(cf.branch[["F_BUS", "T_BUS"]].values.ravel().astype(int))
remap = {}
for row in tx.itertuples():
    src = row.F_BUS if row.T_BUS in used else row.T_BUS
    dest = row.T_BUS if row.T_BUS in used else row.F_BUS
    remap[src] = dest

# --- 3. Merge bus rows ---
sum_cols = [c for c in ["PD", "QD", "GS", "BS"] if c in cf.bus.columns]
for src, dest in remap.items():
    cf.bus.loc[dest, sum_cols] += cf.bus.loc[src, sum_cols].values
    if cf.bus.loc[src, "BUS_TYPE"] == 3:
        cf.bus.loc[dest, "BUS_TYPE"] = 3
cf.bus.drop(index=list(remap.keys()), inplace=True)

# --- 4. Remap generators, fix bus types ---
cf.gen["GEN_BUS"] = cf.gen["GEN_BUS"].astype(int).replace(remap)

In [ ]:
mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()
display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.515345,-12.933126,-30.365960,-3.335616
2,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.634040,-26.664384,60.937556,-7.780875
3,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.062444,-3.078834,-23.994723,-20.381971
4,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-76.005277,-14.618029,76.439472,1.153576
5,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.560528,5.500085,-84.349115,-28.242889
6,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.650885,-21.757111,40.823012,4.453808


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,4,3,0,0,0,0,1,1.040000,-2.216788,345,1,1.1,0.9
2,5,1,90,30,0,0,1,1.039264,-3.744746,345,1,1.1,0.9
3,6,1,0,0,0,0,1,1.077811,1.301106,345,1,1.1,0.9
4,7,1,100,35,0,0,1,1.066897,0.146221,345,1,1.1,0.9
5,8,1,0,0,0,0,1,1.078292,2.846696,345,1,1.1,0.9
6,9,1,125,50,0,0,1,1.025156,-4.006756,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,4,71.338358,-8.479319,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,8,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,6,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# NOTE: to test the reduction:

# cf_ori = copy.deepcopy(cf)

# path = "../tests/data/case6_reduced.m"
# cf = CaseFrames(path, load_case_engine=m)

# mpc = m.runpf(cf.to_mpc(), verbose=False)
# cf = CaseFrames(mpc)
# cf.infer_numpy()

# display(cf.branch)
# display(cf.bus)
# display(cf.gen)

# from matpowercaseframes.testing import assert_frames_struct_equal

# cf_ori._attributes.remove("et")
# cf._attributes.remove("et")
# del(cf_ori.et)
# del(cf.et)

# assert_frames_struct_equal(cf_ori, cf)

In [ ]:
mpc_int = m.ext2int(cf.to_mpc())
[Ybus, Yf, Yt] = m.makeYbus(mpc_int, nout=3)
pd.DataFrame(
    Ybus.todense(),
    columns=cf.bus.index.astype(int),
    index=cf.bus.index.astype(int),
)

bus,1,2,3,4,5,6
bus,,,,,,
1,3.307379-21.947778j,-1.942191+10.510682j,0.000000+ 0.000000j,0.000000+ 0.000000j,0.000000+ 0.000000j,-1.365188+11.604096j
2,-1.942191+10.510682j,3.224200-15.840927j,-1.282009+ 5.588245j,0.000000+ 0.000000j,0.000000+ 0.000000j,0.000000+ 0.000000j
3,0.000000+ 0.000000j,-1.282009+ 5.588245j,2.437097-15.089015j,-1.155087+ 9.784270j,0.000000+ 0.000000j,0.000000+ 0.000000j
4,0.000000+ 0.000000j,0.000000+ 0.000000j,-1.155087+ 9.784270j,2.772210-23.303249j,-1.617122+13.697979j,0.000000+ 0.000000j
5,0.000000+ 0.000000j,0.000000+ 0.000000j,0.000000+ 0.000000j,-1.617122+13.697979j,2.804727-19.445613j,-1.187604+ 5.975135j
6,-1.365188+11.604096j,0.000000+ 0.000000j,0.000000+ 0.000000j,0.000000+ 0.000000j,-1.187604+ 5.975135j,2.552792-17.338230j


In [ ]:
lines_index = cf.branch.index[cf.branch["BR_R"] > 0]

cf.branch_ext = cf.branch.copy()
cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
cf.branch_ext["SFT_MAX"] = (
    cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
).pow(0.5)
cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]

,RATE_A,SFT_MAX,LOADING
branch,,,
1,250,33.142904,0.132572
2,150,66.515976,0.443440
3,150,31.534520,0.210230
4,250,77.824672,0.311299
5,250,91.051556,0.364206
6,250,46.258947,0.185036


## Derating

In [ ]:
def get_branch_thermal_correction_factor(
    t_ambient,
    t_max: float = 90.0,
    t_ref: float = 30.0,
) -> float | pd.Series:
    """
    Compute the ambient temperature correction factor (CF) for conductor ratings.

        CF = sqrt((t_max - t_ambient) / (t_max - t_ref))

    Supports scalar inputs (single CF) or per-branch inputs (Series of CFs),
    allowing each branch to have its own ambient condition and insulation class.

    For a peak hot day with heat wave scenario, this is an overestimating the line
    capability (too small derating) since it assumes:
        - ignore solar heating (overestimate)
        - assume cooling the same (overestimate, heat wave often has low wind, thus
        less cooling)
        - ignore resistive heating from current (overestimate)
        - conductor temperature is at max (underestimate, derating too much for lines
        that are not critically loaded, but fair for critically loaded and overloaded
        lines)
        - ignore radiative cooling (underestimate, radiative cooling increases with
        conductor temperature but is ignored in the formula, but this is likely a small
        effect compared to the others)
    where it overestimate is more significant than underestimate (TODO: proof this).

    Parameters
    ----------
    t_ambient : float or pd.Series
        Actual ambient temperature [°C].
        - float   : same temperature applied to all branches
        - Series  : one value per branch (must match branch DataFrame index)

    t_max : float or pd.Series, optional
        Maximum conductor temperature [°C]. Default 90.0 °C (XLPE insulation).
        - float   : same insulation class for all branches
        - Series  : per-branch insulation class

        Common insulation classes (NEC Table 310.15(B)(1)):
            - 60.0 °C : TW, UF
            - 75.0 °C : THW, THWN, XHHW (wet)
            - 90.0 °C : THWN-2, XHHW-2, USE-2  ← default

    t_ref : float or pd.Series, optional
        Reference ambient temperature of the base ratings [°C]. Default 30.0 °C.
        - NEC NFPA 70 / IEC 60364-5-52 : 30.0 °C  ← default
        - IEEE 835                     : 25.0 °C

    Returns
    -------
    float or pd.Series
        CF >= 0, dimensionless.
        - CF = 1.0 : t_ambient == t_ref     (no derating)
        - CF < 1.0 : t_ambient >  t_ref     (derating required)
        - CF > 1.0 : t_ambient <  t_ref     (uprating permitted)
        - CF = 0.0 : t_ambient >= t_max     (line must not carry current)

    Notes
    -----
    - Formula : Neher & McGrath (1957), IEEE Std 738, NEC §310.15
    - t_max   : NEC Table 310.15(B)(1)
    - t_ref   : NEC NFPA 70 / IEC 60364-5-52

    Read more
    ---------
    https://ieeexplore.ieee.org/document/4499653  # Neher & McGrath (1957)
    https://en.wikipedia.org/wiki/Neher%E2%80%93McGrath_method  # Wikipedia
    https://ecalpro.com/ko/standards/nec/table-310-15-b1-temperature  # NEC Table
    https://ieeexplore.ieee.org/document/10382442  # IEEE Std 738
    https://ieeexplore.ieee.org/document/7111195  # IEEE 835
    https://ieeexplore.ieee.org/document/11030252  # IEEE P848/D1
    """
    return np.sqrt(np.maximum(t_max - t_ambient, 0) / (t_max - t_ref))

In [ ]:
branch_temp = pd.DataFrame(
    {
        "t_ambient": [40.0, 55.0, 45.0, 50.0, 50.0, 38.0],
        "t_max": [90.0, 60.0, 75.0, 90.0, 60.0, 90.0],
        "t_ref": [30.0, 30.0, 30.0, 30.0, 30.0, 30.0],
    },
    index=lines_index,
)

In [ ]:
correction_factors = get_branch_thermal_correction_factor(
    t_ambient=branch_temp["t_ambient"],
    t_max=branch_temp["t_max"],
    t_ref=branch_temp["t_ref"],
)
correction_factors

branch
1    0.912871
2    0.408248
3    0.816497
4    0.816497
5    0.577350
6    0.930949
dtype: float64

In [ ]:
col_rates = ["RATE_A", "RATE_B", "RATE_C"]
cf.branch_ext[col_rates]

,RATE_A,RATE_B,RATE_C
branch,,,
1,250,250,250
2,150,150,150
3,150,150,150
4,250,250,250
5,250,250,250
6,250,250,250


In [ ]:
cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
cf.branch_ext.loc[lines_index, col_rates] = (
    cf.branch_ext.loc[lines_index, col_rates]
    * correction_factors[lines_index].values[:, np.newaxis]
)
cf.branch_ext[col_rates]

,RATE_A,RATE_B,RATE_C
branch,,,
1,228.217732,228.217732,228.217732
2,61.237244,61.237244,61.237244
3,122.474487,122.474487,122.474487
4,204.124145,204.124145,204.124145
5,144.337567,144.337567,144.337567
6,232.737334,232.737334,232.737334


In [ ]:
cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]

,RATE_A,SFT_MAX,LOADING
branch,,,
1,228.217732,33.142904,0.145225
2,61.237244,66.515976,1.086201
3,122.474487,31.534520,0.257478
4,204.124145,77.824672,0.381261
5,144.337567,91.051556,0.630824
6,232.737334,46.258947,0.198760


## Cascading

In [ ]:
# overloaded lines status update
overloaded = cf.branch_ext.loc[:, "LOADING"] > 1.0
cf.branch.loc[overloaded, "BR_STATUS"] = 0

In [ ]:
mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()

cf.branch_ext = cf.branch.copy()
cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
cf.branch_ext["SFT_MAX"] = (
    cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
).pow(0.5)

cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
cf.branch_ext.loc[lines_index, col_rates] = (
    cf.branch_ext.loc[lines_index, col_rates]
    * correction_factors[lines_index].values[:, np.newaxis]
)

cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "F_BUS", "T_BUS", "SFT_MAX", "LOADING"]]

,RATE_A,F_BUS,T_BUS,SFT_MAX,LOADING
branch,,,,,
1,228.217732,4,5,96.250393,0.421748
2,61.237244,5,6,0.000000,0.000000
3,122.474487,6,7,85.690916,0.699663
4,204.124145,7,8,35.515856,0.173991
5,144.337567,8,9,147.616555,1.022717
6,232.737334,9,4,64.363583,0.276550


In [ ]:
# overloaded lines status update
overloaded = cf.branch_ext.loc[:, "LOADING"] > 1.0
cf.branch.loc[overloaded, "BR_STATUS"] = 0

In [ ]:
cf.branch[["F_BUS", "T_BUS", "BR_STATUS"]]

,F_BUS,T_BUS,BR_STATUS
branch,,,
1,4,5,1
2,5,6,0
3,6,7,1
4,7,8,1
5,8,9,0
6,9,4,1


In [ ]:
[groups, isolated] = m.find_islands(cf.to_mpc(), nout=2)
groups, isolated

(Cell([[array([[1., 2., 6.]]), array([[3., 4., 5.]])]]),
 array([], shape=(1, 0), dtype=float64))

In [ ]:
mpcs = m.extract_islands(mpc, groups)

cfs = {}
for isl, mpc in enumerate(mpcs.flatten()):
    cfs[isl] = CaseFrames(mpc)
    cfs[isl].infer_numpy()

    cfs[isl].bus.set_index(cfs[isl].bus["BUS_I"].astype(int), drop=False, inplace=True)
    cfs[isl].bus.index.name = "bus"

    display(cfs[isl].branch[["F_BUS", "T_BUS", "BR_STATUS"]])
    display(cfs[isl].bus)
    display(cfs[isl].gen)

,F_BUS,T_BUS,BR_STATUS
branch,,,
1,4,5,1
2,9,4,1


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
4,4,3,0,0,0,0,1,1.040000,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.001440,-6.569568,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995488,-1.191516,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,4,76.30131,68.332054,300,-300,1.04,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0


,F_BUS,T_BUS,BR_STATUS
branch,,,
1,6,7,1
2,7,8,1


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
6,6,1,0,0,0,0,1,1.011037,16.461747,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.004791,11.622442,345,1,1.1,0.9
8,8,1,0,0,0,0,1,1.023536,12.143127,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,8,163,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
2,6,85,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
cf = cfs[0]
if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
    slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
    slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
    cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus
display(cf.bus)

,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
4,4,3,0,0,0,0,1,1.040000,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.001440,-6.569568,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995488,-1.191516,345,1,1.1,0.9


In [ ]:
cf = cfs[1]
if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
    slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
    slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
    cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus
display(cf.bus)

,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
6,6,1,0,0,0,0,1,1.011037,16.461747,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.004791,11.622442,345,1,1.1,0.9
8,8,3,0,0,0,0,1,1.023536,12.143127,345,1,1.1,0.9


In [ ]:
for _isl, cf in cfs.items():
    if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
        slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
        slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
        cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus

    m.runpf(cf.to_mpc(), verbose=False)
    mpc = m.runpf(cf.to_mpc(), verbose=False)
    cf = CaseFrames(mpc)
    cf.infer_numpy()

    cf.branch_ext = cf.branch.copy()
    cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
    cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
    cf.branch_ext["SFT_MAX"] = (
        cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
    ).pow(0.5)

    lines_index = cf.branch.index[cf.branch["BR_R"] > 0]

    cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
    cf.branch_ext.loc[lines_index, col_rates] = (
        cf.branch_ext.loc[lines_index, col_rates]
        * correction_factors[lines_index].values[:, np.newaxis]
    )

    cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
    display(
        cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]
    )  # see, we can further cascade

,RATE_A,SFT_MAX,LOADING
branch,,,
1,228.217732,96.250393,0.421748
2,102.062073,136.285396,1.335319


,RATE_A,SFT_MAX,LOADING
branch,,,
1,136.930639,85.690917,0.625798
2,102.062073,35.434107,0.347182
